# Task 2b: Specialist Shared Architecture Search (Optuna) & Retraining

Conducts a shared architecture search on representative corruption data as specified in the assignment.
The locked winning architecture is then trained across all three specialists.

### Mathematical Formulation
$$\hat{x}_k = D_k\big(E_k(\tilde{x}_k)\big), \quad \mathcal{L}_k = \alpha\,\mathcal{L}_{L1}(x,\hat{x}_k) + (1-\alpha)\big(1-\mathrm{SSIM}(x,\hat{x}_k)\big)$$

### Optuna Search Space Requirements
As required by the assignment, the shared Optuna search tunes:
1. **Learning Rate** ($\text{lr} \in [10^{-4}, 10^{-3}]$ log scale)
2. **Batch Size** ($B \in \{16, 32, 64\}$)
3. **Bottleneck Dimension** ($\text{latent\_dim} \in \{64, 128, 256\}$)
4. **Channel Configuration** ($\text{base\_channels} \in \{32, 64\}$)
5. **Dropout Rate** ($\text{dropout} \in [0.0, 0.3]$)
6. **L1-to-SSIM Loss Weighting** ($\alpha \in [0.5, 0.95]$)

### Retraining:
The winning shared architecture is trained independently on each of the 3 corruptions for `FULL_TRAIN_EPOCHS = 35` to produce `task2b_specialist_{ctype}_final_best.pt`.

In [1]:
# 1. Study Configuration
N_TRIALS = 20
OPTUNA_EPOCHS = 15
FULL_TRAIN_EPOCHS = 35
STUDY_NAME = "task2b_specialists"
STORAGE_DB = "sqlite:///task2b_study.db"
SEED = 42

In [2]:
# 2. Imports & Setup
import sys
from pathlib import Path
import yaml
import torch
from torch.utils.data import DataLoader
import optuna

RESEARCH_ROOT = Path("..").resolve()
if str(RESEARCH_ROOT) not in sys.path:
    sys.path.insert(0, str(RESEARCH_ROOT))

from constants import CONFIGS_ROOT
from src.device_utils import get_device, device_report
from src.datasets import PetDataset
from src.models_dae import ConvDAE
from src.train_loop import train_one_epoch_dae, validate_dae, set_seed
from src.checkpoint_utils import create_checkpoint_dict, save_checkpoint

set_seed(SEED)
device = get_device()

/home/zaifi/genai-assignment-1/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[device] Using GPU via CUDA: NVIDIA GeForce RTX 5050 Laptop GPU


In [3]:
# 3. Shared Architecture Objective
train_dataset = PetDataset(mode="train", corruption_mode="salt", seed=SEED)
val_dataset = PetDataset(mode="val", corruption_mode="salt", seed=SEED)

def objective(trial: optuna.Trial) -> float:
    lr = trial.suggest_float("lr", 1e-4, 1e-3, log=True)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])
    latent_dim = trial.suggest_categorical("latent_dim", [64, 128, 256])
    base_channels = trial.suggest_categorical("base_channels", [32, 64])
    dropout = trial.suggest_float("dropout", 0.0, 0.3)
    alpha = trial.suggest_float("alpha", 0.5, 0.95)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    model = ConvDAE(base_channels=base_channels, latent_dim=latent_dim,
                    dropout=dropout, use_skip=False).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    best_loss = float("inf")
    for epoch in range(1, OPTUNA_EPOCHS + 1):
        train_one_epoch_dae(model, train_loader, optimizer, device, alpha=alpha)
        metrics = validate_dae(model, val_loader, device, alpha=alpha)
        best_loss = min(best_loss, metrics["val_loss"])
        trial.report(metrics["val_loss"], step=epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return best_loss


In [ ]:
# 4. Run Study & Export Configs
study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=STORAGE_DB,
    load_if_exists=True,
    direction="minimize",
    pruner=optuna.pruners.MedianPruner()
)
study.optimize(objective, n_trials=N_TRIALS)

print(f"Winning Shared Architecture Params: {study.best_params}")

for ctype in ["salt", "blur", "occlusion"]:
    cfg_path = CONFIGS_ROOT / f"task2b_{ctype}_best_config.yaml"
    with open(cfg_path, "w") as f:
        yaml.dump({
            "task": f"task2b_specialist_{ctype}",
            "params": study.best_params,
            "best_val_loss": float(study.best_value)
        }, f, indent=2)
print("Saved configs for all 3 specialists.")

In [5]:
# 5. Retrain All 3 Specialists on Full Schedule with Winning Architecture
p = study.best_params
print(f"\n--- Retraining Specialists with Winning Architecture ({FULL_TRAIN_EPOCHS} Epochs) ---")
print(p)

checkpoint_dir = RESEARCH_ROOT / "checkpoints"

for ctype in ["salt", "blur", "occlusion"]:
    print(f"\n[Retraining Specialist: {ctype}]")
    train_ds = PetDataset(mode="train", corruption_mode=ctype, seed=SEED)
    val_ds = PetDataset(mode="val", corruption_mode=ctype, seed=SEED)
    train_loader = DataLoader(train_ds, batch_size=p["batch_size"], shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=p["batch_size"], shuffle=False, num_workers=2)
    
    model = ConvDAE(
        base_channels=p["base_channels"],
        latent_dim=p["latent_dim"],
        dropout=p["dropout"],
        use_skip=False
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=p["lr"])
    scaler = torch.amp.GradScaler("cuda") if device.type == "cuda" else None
    best_loss = float("inf")
    
    for epoch in range(1, FULL_TRAIN_EPOCHS + 1):
        tr = train_one_epoch_dae(model, train_loader, optimizer, device, alpha=p["alpha"], scaler=scaler)
        vl = validate_dae(model, val_loader, device, alpha=p["alpha"])
        is_best = vl["val_loss"] < best_loss
        best_loss = min(best_loss, vl["val_loss"])
        ckpt = create_checkpoint_dict(
            model, optimizer, epoch, vl["val_loss"], random_seed=SEED,
            optuna_trial_number=int(study.best_trial.number),
            extra_metadata={**p, "corruption_type": ctype}
        )
        save_checkpoint(
            checkpoint_dir=checkpoint_dir,
            prefix=f"task2b_specialist_{ctype}_final",
            ckpt_dict=ckpt,
            is_best=is_best,
            task_name=f"task2b_{ctype}",
            phase_name="final",
            device_name=device_report()["device_name"]
        )
        if epoch % 5 == 0 or epoch == FULL_TRAIN_EPOCHS:
            print(f"[{ctype} {epoch:02d}/{FULL_TRAIN_EPOCHS:02d}] Train: {tr['train_loss']:.4f} | Val: {vl['val_loss']:.4f} (SSIM: {vl['val_ssim']:.4f}) {'*Best*' if is_best else ''}")

print("\nAll 3 specialists retrained and exported successfully.")


--- Retraining Specialists with Winning Architecture (35 Epochs) ---
{'lr': 0.00010676565989378756, 'batch_size': 32, 'latent_dim': 256, 'base_channels': 64, 'dropout': 0.026025700426524835, 'alpha': 0.9471441383762044}

[Retraining Specialist: salt]
[salt 05/35] Train: 0.1850 | Val: 0.1824 (SSIM: 0.3903) *Best*
[salt 10/35] Train: 0.1773 | Val: 0.1734 (SSIM: 0.3985) *Best*
[salt 15/35] Train: 0.1700 | Val: 0.1735 (SSIM: 0.3997) 
[salt 20/35] Train: 0.1658 | Val: 0.1672 (SSIM: 0.4049) 
[salt 25/35] Train: 0.1590 | Val: 0.1728 (SSIM: 0.4078) 
[salt 30/35] Train: 0.1564 | Val: 0.1573 (SSIM: 0.4164) 
[salt 35/35] Train: 0.1529 | Val: 0.1514 (SSIM: 0.4244) 

[Retraining Specialist: blur]
[blur 05/35] Train: 0.1828 | Val: 0.1757 (SSIM: 0.3953) *Best*
[blur 10/35] Train: 0.1750 | Val: 0.1669 (SSIM: 0.4037) *Best*
[blur 15/35] Train: 0.1659 | Val: 0.1640 (SSIM: 0.4059) *Best*
[blur 20/35] Train: 0.1616 | Val: 0.1573 (SSIM: 0.4128) *Best*
[blur 25/35] Train: 0.1529 | Val: 0.1551 (SSIM: 0.4202